# 03 - pooling ablation

Hypothesis from notebook 02: mean-pooling over 109 positions **dilutes** a signal concentrated
near the 3' end, which is why L12-mean was the only arm that failed to gain from the downstream
window.

Window layout for the 109 nt case:

```
token   0        <cls>
tokens  1..90    upstream, ending AT the termination site
tokens 91..109   the 19 nt downstream
token  110       <eos>
```

Everything below uses frozen embeddings, identical folds, target `ln(Ts)`.

In [1]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, wilcoxon
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.simplefilter("ignore")
REPO = Path.cwd().parent if Path.cwd().name == "probing" else Path.cwd()
EMB, RES = REPO / "probing/embeddings", REPO / "probing/results"
ALPHAS = np.logspace(-3, 6, 40)

FOLDS = json.loads((RES / "folds.json").read_text())
print("folds:", FOLDS["n_folds"], "| seed:", FOLDS["seed"], "| schemes:", list(FOLDS["schemes"]))

folds: 10 | seed: 42 | schemes: ['KFold', 'GroupKFold']


In [2]:
def per_fold(make_model, X, y, folds):
    """Fit on each training fold, score the held-out fold. Returns (r2 array, rho array)."""
    r2, rho = [], []
    for tr, te in folds:
        tr, te = np.array(tr), np.array(te)
        m = make_model()
        m.fit(X[tr], y[tr])
        p = m.predict(X[te])
        r2.append(r2_score(y[te], p))
        rho.append(spearmanr(y[te], p).statistic)
    return np.array(r2), np.array(rho)

RIDGE = lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=ALPHAS))

def one_hot(seqs, L):
    idx = {c: i for i, c in enumerate("ACGU")}
    M = np.zeros((len(seqs), L, 4), dtype=np.float32)
    for i, s in enumerate(seqs):
        for j, c in enumerate(s):
            M[i, j, idx[c]] = 1.0
    return M.reshape(len(seqs), -1)

In [3]:
L, UP = 109, 90
df = pd.read_csv(REPO/"probing/data/chen2013_window_up90_dn19.csv").rename(columns={"window":"sequence","strength":"Ts"})
y = np.log(df.Ts.to_numpy())

def poolings(E):
    nt   = E[:, 1:L+1, :]        # nucleotides only
    upst = E[:, 1:UP+1, :]       # through the TTS
    down = E[:, UP+1:L+1, :]     # the 19 nt downstream
    return {
        "mean (all)":            nt.mean(1),
        "max (all)":             nt.max(1),
        "mean+max":              np.hstack([nt.mean(1), nt.max(1)]),
        "<cls>":                 E[:, 0, :],
        "mean upstream only":    upst.mean(1),
        "mean downstream only":  down.mean(1),
        "upstream+downstream":   np.hstack([upst.mean(1), down.mean(1)]),
        "mean last 30":          nt[:, -30:, :].mean(1),
        "mean last 10":          nt[:, -10:, :].mean(1),
        "flatten all positions": nt.reshape(len(E), -1),   # free positional weighting
    }
print("target ln(Ts) | n =", len(y))

target ln(Ts) | n = 582


In [4]:
rows = []
for layer in [6, 12]:
    E = np.load(EMB / f"rnafm_L{L}_layer{layer}_tokens.npy")
    P = poolings(E)
    for scheme, folds in FOLDS["schemes"].items():
        base = None
        print(f"=== layer {layer} | {scheme} ===")
        print(f"{'pooling':<24}{'dim':>8}{'R2':>9}{'sd':>8}{'vs mean':>10}{'wins':>7}{'p':>8}")
        for name, X in P.items():
            r, _ = per_fold(RIDGE, X, y, folds)
            if name == "mean (all)":
                base = r
                print(f"{name:<24}{X.shape[1]:>8}{r.mean():>+9.3f}{r.std():>8.3f}{'--':>10}{'--':>7}{'--':>8}")
            else:
                d = r - base
                print(f"{name:<24}{X.shape[1]:>8}{r.mean():>+9.3f}{r.std():>8.3f}"
                      f"{d.mean():>+10.3f}{int((d>0).sum()):>5}/10{wilcoxon(base, r).pvalue:>8.3f}")
            rows.append(dict(layer=layer, cv=scheme, pooling=name, dim=X.shape[1],
                             r2_mean=r.mean(), r2_sd=r.std()))
        print()
    del P, E
abl = pd.DataFrame(rows)

=== layer 6 | KFold ===
pooling                      dim       R2      sd   vs mean   wins       p


mean (all)                   640   +0.713   0.048        --     --      --


max (all)                    640   +0.723   0.056    +0.011    5/10   0.492


mean+max                    1280   +0.736   0.054    +0.024    9/10   0.004


<cls>                        640   +0.714   0.047    +0.001    6/10   0.922


...

mean downstream only         640   +0.609   0.130    +0.019    7/10   0.193


upstream+downstream         1280   +0.611   0.130    +0.022    8/10   0.193


mean last 30                 640   +0.639   0.133    +0.050   10/10   0.002


mean last 10                 640   +0.621   0.133    +0.032    8/10   0.084


flatten all positions      69760   +0.626   0.112    +0.036    6/10   0.193



In [5]:
print("top 5 by GroupKFold R2:")
print(abl[abl.cv=="GroupKFold"].sort_values("r2_mean", ascending=False).head(5)
      .to_string(index=False, float_format=lambda v: f"{v:+.4f}"))

top 5 by GroupKFold R2:
 layer         cv             pooling  dim  r2_mean   r2_sd
     6 GroupKFold            mean+max 1280  +0.6663 +0.1282
     6 GroupKFold           max (all)  640  +0.6519 +0.1305
     6 GroupKFold upstream+downstream 1280  +0.6463 +0.1238
    12 GroupKFold        mean last 30  640  +0.6390 +0.1327
     6 GroupKFold        mean last 10  640  +0.6386 +0.1197


**Reading:** the dilution hypothesis holds for **layer 12** (restricting to the last 30
positions gains a lot) but is **false for layer 6**, which instead wants a second summary
statistic over the same positions (`mean+max`).

Two negative results worth keeping: `flatten all positions` (69,760 features, free positional
weighting) does **not** win - ridge cannot exploit that freedom at n=582, so the implicit
averaging in pooling is acting as useful regularisation. And `mean downstream only` scores as
well as the full window at layer 6, i.e. those 19 positions already encode the upstream through
self-attention.

**Caveat:** 10 poolings x 2 layers x 2 schemes = 40 configurations, and the best was selected
post hoc. With fold sd around 0.13, re-run the winner on a fresh fold seed before relying on it.